# drfn clean grid: ByT5 analyser labels (seed 11)

Cleaned AGE text (invisible zero-width characters removed from Amharic,
punctuation detached from words), a BPE that keeps word boundaries, and
`gmm` with the balance regulariser off. Details: `build_report.json`.

| run | what it is |
|---|---|
| `bpe_aux` | labels on two side heads |
| `cmm_spec` | the section 3.4 CMM plus side heads; best translator on the old data |
| `gmm` | labels supervise the memory's own routing (grounded) |

After the last epoch each run decodes the full dev set from `best.pt` and
writes `dev_hyps.json` and `final_eval.json` (character chrF on full dev,
surface-form routing probes at root and pattern level), so the results can be
fetched without weights.

In [ ]:
SEED = 11
LABELS = "analyser"
VARIANTS = "bpe_aux cmm_spec gmm"

In [ ]:
!pip install -q tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_clean.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
print('pack', pack_src)
def stream(cmd, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if line.startswith('{"step"') or line.startswith("{'step'"): continue
        print(line, end='')
    return p.wait()

In [ ]:
env = dict(os.environ, SEED=str(SEED), LABELS=LABELS, VARIANTS=VARIANTS)
print('exit', stream(['bash', 'run_clean.sh'], env=env))

In [ ]:
rows = []
for d in sorted(glob.glob('runs/*_s*')):
    lg, fe = os.path.join(d, 'train_log.jsonl'), os.path.join(d, 'final_eval.json')
    if not os.path.exists(lg): continue
    ev = [json.loads(l) for l in open(lg) if '"dev_loss"' in l]
    if not ev: continue
    b = min(ev, key=lambda e: e['dev_loss'])
    f = json.load(open(fe)) if os.path.exists(fe) else {}
    pr = (f.get('probe_root') or {}).get('all_types') or {}
    pp = (f.get('probe_pattern') or {}).get('all_types') or {}
    rows.append((os.path.basename(d), len(ev), b['dev_loss'], b['epoch'],
                 f.get('chrF_char_full_dev'), pr.get('consistency'), pr.get('permutation'),
                 (f.get('probe_root') or {}).get('unseen_slot_accuracy'),
                 pp.get('consistency'), pp.get('permutation')))
fmt = lambda v, s='%.3f': 'n/a' if v is None else s % v
print('%-26s %4s %9s %4s %10s %15s %10s %15s' % ('run', 'ep', 'best loss', '@ep',
      'chrF full', 'root vs chance', 'root unseen', 'pattern vs chance'))
for r in rows:
    print('%-26s %4d %9.4f %4d %10s %15s %10s %15s' % (r[0], r[1], r[2], r[3], fmt(r[4], '%.2f'),
          fmt(r[5]) + '/' + fmt(r[6]), ('%.1f%%' % (100 * r[7])) if r[7] is not None else 'n/a',
          fmt(r[8]) + '/' + fmt(r[9])))
print('\nchrF full = character chrF on the FULL dev set (1,741 sentences), invisible characters removed.')
print('Not comparable with earlier numbers: the data and tokenizer changed. Compare runs within this grid.')
open('/kaggle/working/summary.txt', 'w').write(repr(rows))